In [7]:
!pip install google-genai
!pip install pinecone
!pip install rapidfuzz
!pip install tqdm
!pip install ipywidgets


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [18]:
# ==============================================================================
# 💡 LESSONS LEARNED
# ==============================================================================
#    NO GUARANTEED BATCH ORDER IN LLM APIS:
#    When sending multiple PDFs to Gemini in a single request,
#    the returned outputs are NOT guaranteed to preserve input order. In addition,
#    documents may be skipped, merged, or inconsistently processed due to token
#    limits, safety filters, or internal model behavior.
#    Even when explicitly instructing the model to return a stable identifier
#    the LLM may hallucinate, duplicate, or misassign identifiers.
#    Therefore, model-generated IDs cannot be treated as a source of
#    truth for data alignment. Processing each CV independently
#    ensures deterministic mapping between input document and output artifacts.
#    This eliminates dependency on ordering, batching behavior, or LLM adherence.
#    For performance scaling, concurrency should be implemented at the application
#    level (e.g. asyncio). This preserves deterministic per-document processing
#    while still enabling high throughput.
#
#    DENSE VECTORS ARE SEMANTICALLY FORGIVING:
#    Normalization is not strictly required for semantic search.
#    Models like 'gemini-embedding-2' naturally understand that 'React.js',
#    'ReactJS' and 'React' occupy the same neighborhood in the embedding space.
#    The dense search handles these synonyms automatically without extra logic.
# ==============================================================================

from google import genai
from google.genai import types
from pinecone import Pinecone
from pydantic import BaseModel, Field
from tqdm import tqdm
from datetime import date
from enum import Enum
from rapidfuzz import process, fuzz
from dotenv import load_dotenv
import csv
import time
import random
import uuid
import os

load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")

In [14]:
def load_taxonomy_skills(filepath: str) -> list[str]:
    skills = []
    try:
        with open(filepath, mode='r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            for row in reader:
                label = row.get('preferredLabel')
                if label:
                    skills.append(label)
    except Exception as e:
        print(f"Error loading taxonomy: {e}")

    return skills

def normalize_skills(raw_skills: list[str], taxonomy_skills: list[str], threshold: float = 80.0) -> list[str]:
    normalized = []
    for skill in raw_skills:
        best_match = process.extractOne(skill, taxonomy_skills, scorer=fuzz.WRatio)
        if best_match:
            match_string, match_score, _ = best_match
            if match_score >= threshold:
                normalized.append(match_string)
            else:
                normalized.append(skill)
    return list(set(normalized))

In [16]:
gemini_client = genai.Client()
pinecone_client = Pinecone()
pinecone_index = pinecone_client.Index('cjm-semantic')
index_namespace = "matching-pool"

class DocType(str, Enum):
    CV = "cv"
    VACANCY = "vacancy"

class CandidateMetadata(BaseModel):
    country: str = Field(description="Two character country code where the candidate resides, e.g. 'DE', 'CH', 'AT', 'US'. If unclear, set to 'UNKNOWN'")
    city: str = Field(description="The candidate's city of residence, e.g. 'Berlin', 'Bern', 'Vienna', 'New York'. If unclear, set to 'UNKNOWN'")
    career_level: str = Field(description="The candidates career level. Must be exactly one of these values: 'Junior', 'Mid', 'Senior', 'Lead'. If unclear, set to 'UNKNOWN'")
    department: str = Field(description="The candidates broad job department, e.g. 'Software Development', 'Data Science', 'Solution Architect', 'Tester'. If unclear, set to 'UNKNOWN'")
    raw_skills: list[str] = Field(description="A list of the core competencies of the candidate, e.g. tech stacks, tools, or methods. If unclear, return ['UNKNOWN']")
    highest_degree: str = Field(description="The highest level of education the candidate completed, e.g. 'Bachelor', 'Master', 'PhD', 'Apprenticeship'. If unclear, set to 'UNKNOWN'")
    salary_amount: int = Field(description="Expected annual gross salary of the candidate. Must be a plain integer. If unclear, set to 0")
    salary_currency: str = Field(description="ISO currency code of expected annual gross salary of the candidate, e.g. 'CHF', 'EUR', 'USD'. If unclear, 'UNKNOWN'")

class VacancyMetadata(BaseModel):
    country: str = Field(description="Two character country code where the vacancy is located, e.g. 'DE', 'CH', 'AT', 'US'. If unclear, set to 'UNKNOWN'")
    city: str = Field(description="The city where the vacancy is located, e.g. 'Berlin', 'Bern', 'Vienna', 'New York'. If unclear, set to 'UNKNOWN'")
    career_level: str = Field(description="Target career level for the vacancy. Must be exactly one of these values: 'Junior', 'Mid', 'Senior', 'Lead'. If unclear, set to 'UNKNOWN'")
    department: str = Field(description="Broad job department the the vacancy can be assigned to, e.g. 'Software Development', 'Data Science', 'Solution Architect', 'Tester'. If unclear, set to 'UNKNOWN'")
    raw_skills: list[str] = Field(description="A list of the critical core competencies required for the vacancy, e.g. tech stacks, tools, or methods. If unclear, return ['UNKNOWN']")
    highest_degree: str = Field(description="Minimum education required for the vacancy, e.g. 'Bachelor', 'Master', 'PhD', 'Apprenticeship'. If unclear, set to 'UNKNOWN'")
    salary_amount: int = Field(description="Maximum annual gross salary for the vacancy. Must be a plain integer. If unclear, set to 0")
    salary_currency: str = Field(description="ISO currency code of maximum annual gross salary for the vacancy, e.g. 'CHF', 'EUR', 'USD'. If unclear, 'UNKNOWN'")

cv_prompt = """
    You are an HR data extraction system.

    Extract structured information from the CV according to the provided schema.

    Rules:
    - Use ONLY information explicitly stated in the CV
    - Do NOT infer or guess missing data

    Return a valid JSON matching the schema. No explanations required.
    """

vacancy_prompt = """
    You are an HR data extraction system.

    Extract structured information from the vacancy according to the provided schema.

    Rules:
    - Use ONLY information explicitly stated in the vacancy
    - Do NOT infer or guess missing data

    Return a valid JSON matching the schema. No explanations required.
    """

def process_documents(folder_path: str, prompt: str, doc_type: DocType, metadata_schema: type[BaseModel], taxonomy_skills: list[str]):
    if not os.path.exists(folder_path):
        print(f"Specified folder path '{folder_path}' does not exist")
        return

    vectors_to_upsert = []

    files = [fn for fn in sorted(os.listdir(folder_path)) if fn.lower().endswith('.pdf')]
    pbar = tqdm(files, desc=f"Processing {doc_type}", unit=doc_type)

    for fn in pbar:
            try:
                with open(os.path.join(folder_path, fn), "rb") as f:
                    pdf_part = types.Part.from_bytes(data=f.read(), mime_type='application/pdf')

                inferred_metadata = retry_call(lambda: gemini_client.models.generate_content(
                model='gemini-3.5-flash',
                contents=[pdf_part, prompt],
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=metadata_schema,
                    temperature=0.0
                    ),
                ))

                validated_metadata = metadata_schema.model_validate_json(inferred_metadata.text).model_dump()
                validated_metadata["normalized_skills"] = normalize_skills(validated_metadata.get("raw_skills", []), taxonomy_skills)
            except Exception as e:
                pbar.set_postfix(stage="error")
                print(f"Error whilst inferring metadata {fn}: {e}")
                continue

            try:
                inferred_dense_embedding = retry_call(lambda: gemini_client.models.embed_content(
                    model="gemini-embedding-2",
                    contents=[pdf_part],
                    config=types.EmbedContentConfig(
                        output_dimensionality=768,
                        task_type = "RETRIEVAL_DOCUMENT" if doc_type == DocType.CV else "RETRIEVAL_QUERY"
                    )
                ))
            except Exception as e:
                pbar.set_postfix(stage="error")
                print(f"Error whilst embedding {fn}: {e}")
                continue

            vectors_to_upsert.append({
                "id": str(uuid.uuid4()),
                "values": inferred_dense_embedding.embeddings[0].values,
                "metadata": {
                    "type": doc_type,
                    "filename": fn,
                    "last_updated": date.today().isoformat(),
                    **validated_metadata
                }
            })

    for i in range(0, len(vectors_to_upsert), 25):
        pinecone_index.upsert(vectors=vectors_to_upsert[i:i + 25], namespace=index_namespace)
    pbar.close()

def retry_call(fn, max_retries=3, base_delay=1.0):
    for attempt in range(max_retries):
        try:
            return fn()

        except Exception as e:
            wait = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
            print(f"[RETRY] attempt {attempt+1}/{max_retries} failed: {e}. retrying in {wait:.2f}s")
            time.sleep(wait)

    raise RuntimeError(f"max retries of {max_retries} exceeded")

In [17]:
taxonomy_skills = load_taxonomy_skills("taxonomy/skills_en.csv")

process_documents(
    folder_path="cv",
    doc_type="cv",
    metadata_schema=CandidateMetadata,
    prompt=cv_prompt,
    taxonomy_skills=taxonomy_skills
)

process_documents(
    folder_path="vacancy",
    doc_type="vacancy",
    metadata_schema=VacancyMetadata,
    prompt=vacancy_prompt,
    taxonomy_skills=taxonomy_skills
)

Processing vacancy: 100%|██████████| 1/1 [00:10<00:00, 10.12s/vacancy]


In [5]:
vacancies = pinecone_index.query(
    namespace=index_namespace,
    vector=[0.0] * 768,
    top_k=1,
    include_metadata=True,
    include_values=True,
    filter={"type": {"$eq": "vacancy"}}
)

vacancy = vacancies["matches"][0]
vacancy_dense_embedding = vacancy["values"]
vacancy_metadata = vacancy["metadata"]

matched_candidates = pinecone_index.query(
    namespace=index_namespace,
    vector=vacancy_dense_embedding,
    top_k=8,
    include_metadata=True,
    filter={
            "$and": [
                {"type": {"$eq": "cv"}},
                {"country": {"$in": [vacancy_metadata.get("country"),"GB"]}},
                {"department": {"$eq": vacancy_metadata.get("department")}},
                {"career_level": {"$in": [vacancy_metadata.get("career_level"),"Mid"]}},
                {"salary_amount": {"$lte": int(vacancy_metadata.get("salary_amount", 0))}},
            ]
        }
)

for candidate in matched_candidates["matches"]:
        score = candidate["score"]
        metadata = candidate["metadata"]
        print(f"Candidate: {metadata.get('filename')}")
        print(f"Score: {score}")

Candidate: director_it.pdf
Score: 0.645709515
Candidate: senior_system_administrator.pdf
Score: 0.636615574
Candidate: senior_data_scientist.pdf
Score: 0.629840553
Candidate: senior_culinary_chef.pdf
Score: 0.620315135
Candidate: junior_developer.pdf
Score: 0.606776834
Candidate: junior_business_analyst.pdf
Score: 0.600215316
Candidate: senior_consultant.pdf
Score: 0.593650341
Candidate: senior_infrastructure_architect.pdf
Score: 0.584743261
